# Compare M0–M4 micro-search results with the shared Stage 1 spatial reference

Load exactly one standardized summary for each candidate. The notebook checks the common Stage 0 and Stage 1 hashes before applying the prespecified screening heuristics. It does not compute a composite score or claim statistical significance.


In [1]:
# Attach/extract Stage 1 and M0-M4 bundles, or set explicit result directories.
STAGE1_RESULT_DIR = "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/stage1_shared_micro_e1"
RESULT_DIRS = {"M0": "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/M0_global_dct_gap", 
               "M1": "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/M1_dct_pool4x4", 
               "M2": "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/M2_coord_dct_pool4x4", 
               "M3": "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/M3_band_aware_dct", 
               "M4": "/kaggle/input/datasets/maithanhphuong/micro-summaries-bundle-full/micro_summaries_bundle/M4_block_dct_8x8"}
OUTPUT_DIR = "/kaggle/working/pad_micro_search_v1/comparison"


In [2]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

CANDIDATES = ["M0", "M1", "M2", "M3", "M4"]
CANDIDATE_SCHEMA = "pad_micro_candidate_summary_v1"
STAGE1_SCHEMA = "pad_micro_stage1_summary_v1"
REQUIRED_BRANCH_MODES = {"FULL", "FREQ_ZERO", "FREQ_SHUFFLE", "SPATIAL_ZERO"}


def reject_json_constant(value):
    raise ValueError("Non-standard JSON numeric constant: {}".format(value))


def read_json_strict(path):
    return json.loads(Path(path).read_text(), parse_constant=reject_json_constant)


def matching_paths(filename, explicit_dir=""):
    if explicit_dir:
        root = Path(explicit_dir).expanduser().resolve()
        if root.is_file():
            return [root] if root.name == filename else []
        if not root.is_dir():
            raise FileNotFoundError("Configured result directory does not exist: {}".format(root))
        return sorted(path.resolve() for path in root.rglob(filename) if path.is_file())
    root = Path("/kaggle/input")
    return sorted(path.resolve() for path in root.rglob(filename) if path.is_file()) if root.exists() else []


def resolve_stage1_summary():
    matches = matching_paths("micro_e1_v1_summary.json", STAGE1_RESULT_DIR)
    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one Stage 1 micro_e1_v1_summary.json, found {}. "
            "Set STAGE1_RESULT_DIR explicitly. Matches: {}".format(len(matches), [str(x) for x in matches])
        )
    summary = read_json_strict(matches[0])
    required = {"resource_hashes", "checkpoint_sha256", "val", "lcc", "lcc_candidate_n", "lcc_valid_face_n"}
    if summary.get("schema") != STAGE1_SCHEMA or not required.issubset(summary):
        raise RuntimeError("Stage 1 summary has an unsupported schema or missing required fields: {}".format(matches[0]))
    if not isinstance(summary["resource_hashes"], dict) or not summary["checkpoint_sha256"]:
        raise RuntimeError("Stage 1 summary is missing checkpoint/resource identity")
    return matches[0], summary


def resolve_candidate_summary(candidate_id):
    paths = matching_paths("summary.json", RESULT_DIRS.get(candidate_id, ""))
    valid = []
    invalid = []
    for path in paths:
        try:
            data = read_json_strict(path)
        except Exception as exc:
            invalid.append((str(path), str(exc)))
            continue
        if data.get("candidate_id") == candidate_id:
            valid.append(path)
    if len(valid) != 1:
        raise RuntimeError(
            "Expected exactly one {} summary.json, found {}. Set RESULT_DIRS[{}]. "
            "Matches: {}".format(candidate_id, len(valid), candidate_id, [str(x) for x in valid or paths])
        )
    summary = read_json_strict(valid[0])
    required = {"resource_hashes", "frequency_only", "concat", "branch_ablation", "feature_diagnostics", "params", "budget"}
    if summary.get("schema") != CANDIDATE_SCHEMA or not required.issubset(summary):
        raise RuntimeError("{} summary has an unsupported schema or missing required fields".format(candidate_id))
    return valid[0], summary


def finite_value(value, name):
    try:
        value = float(value)
    except (TypeError, ValueError) as exc:
        raise RuntimeError("Missing or invalid metric {}".format(name)) from exc
    if not math.isfinite(value):
        raise RuntimeError("Metric {} is non-finite".format(name))
    return value


def finite_metric(summary, branch, domain, name):
    try:
        value = summary[branch][domain][name]
    except (KeyError, TypeError) as exc:
        raise RuntimeError("Missing metric {}.{}.{}".format(branch, domain, name)) from exc
    return finite_value(value, "{}.{}.{}".format(branch, domain, name))


def metric_relation(delta, tolerance, higher_is_better=True):
    if abs(delta) <= tolerance:
        return "WITHIN_DESCRIPTIVE_TOLERANCE"
    improved = delta > 0 if higher_is_better else delta < 0
    return "BETTER" if improved else "WORSE"


stage1_path, stage1 = resolve_stage1_summary()
stage1_hashes = stage1["resource_hashes"]
stage1_sha = str(stage1["checkpoint_sha256"])
if not stage1_sha or stage1_hashes.get("shared_micro_e1_checkpoint_sha256") not in (None, stage1_sha):
    raise RuntimeError("Stage 1 summary checkpoint SHA is inconsistent")
spatial_val_acer = finite_value(stage1["val"].get("acer"), "Stage1 Val ACER")
spatial_val_auc = finite_value(stage1["val"].get("auc"), "Stage1 Val AUC")
spatial_lcc_auc = finite_value(stage1["lcc"].get("auc"), "Stage1 LCC AUC")
spatial_lcc_hter = finite_value(stage1["lcc"].get("hter"), "Stage1 LCC HTER")

summaries = {}
summary_paths = {"Stage1": str(stage1_path)}
shuffle_fingerprints = {"val": set(), "lcc": set()}
for candidate_id in CANDIDATES:
    path, summary = resolve_candidate_summary(candidate_id)
    summaries[candidate_id] = summary
    summary_paths[candidate_id] = str(path)
    hashes = dict(summary["resource_hashes"])
    candidate_stage1_sha = hashes.pop("shared_micro_e1_checkpoint_sha256", None)
    if candidate_stage1_sha != stage1_sha:
        raise RuntimeError("{} used a different shared Stage 1 checkpoint SHA".format(candidate_id))
    if hashes != stage1_hashes:
        raise RuntimeError("{} Stage 0 resource hashes/counts differ from Stage 1".format(candidate_id))
    if summary["budget"] != summaries.get("M0", summary)["budget"]:
        raise RuntimeError("{} used a different micro training budget".format(candidate_id))
    if set(summary["branch_ablation"].get("val", {})) != REQUIRED_BRANCH_MODES:
        raise RuntimeError("{} Val branch modes are incomplete".format(candidate_id))
    if set(summary["branch_ablation"].get("lcc", {})) != REQUIRED_BRANCH_MODES:
        raise RuntimeError("{} LCC branch modes are incomplete".format(candidate_id))
    for branch, domain in [("frequency_only", "val"), ("frequency_only", "lcc"), ("concat", "val"), ("concat", "lcc")]:
        for name in ["acer", "auc", "apcer", "bpcer", "hter"]:
            finite_metric(summary, branch, domain, name)
    for mode in REQUIRED_BRANCH_MODES:
        for name in ["auc", "hter", "apcer", "bpcer"]:
            value = summary["branch_ablation"]["lcc"][mode].get(name)
            if value is None or not math.isfinite(float(value)):
                raise RuntimeError("{} LCC branch metric {}.{} is missing/non-finite".format(candidate_id, mode, name))
    for domain, expected_n in [("val", 2000), ("lcc", int(stage1["lcc_valid_face_n"]))]:
        diag = summary["feature_diagnostics"].get(domain, {})
        if int(diag.get("diagnostic_shuffle_seed", -1)) != 20260929:
            raise RuntimeError("{} {} shuffle seed differs from the frozen diagnostic policy".format(candidate_id, domain))
        if int(diag.get("fixed_point_count", -1)) != 0 or int(diag.get("permutation_n", -1)) != expected_n:
            raise RuntimeError("{} {} shuffle permutation is not a full derangement".format(candidate_id, domain))
        if not isinstance(diag.get("permutation_sha256"), str) or len(diag["permutation_sha256"]) != 64:
            raise RuntimeError("{} {} shuffle permutation SHA256 is missing".format(candidate_id, domain))
        shuffle_fingerprints[domain].add(diag["permutation_sha256"])
    expected_candidate_n = int(stage1["lcc_candidate_n"])
    expected_valid_n = int(stage1["lcc_valid_face_n"])
    for branch in ["frequency_only", "concat"]:
        lcc_metrics = summary[branch]["lcc"]
        if int(lcc_metrics.get("candidate_n", -1)) != expected_candidate_n or int(lcc_metrics.get("n_scored", -1)) != expected_valid_n:
            raise RuntimeError("{} {} did not evaluate the common Stage 0 LCC set".format(candidate_id, branch))

if any(len(values) != 1 for values in shuffle_fingerprints.values()):
    raise RuntimeError("M0-M4 did not use the same deterministic FREQ_SHUFFLE permutation per evaluation domain")

reference = summaries["M0"]
reference_hashes = reference["resource_hashes"]
reference_budget = reference["budget"]
for candidate_id, summary in summaries.items():
    if summary["resource_hashes"] != reference_hashes or summary["budget"] != reference_budget:
        raise RuntimeError("{} does not share M0 Stage 0/Stage 1 resources and budget".format(candidate_id))

m0_auc = finite_metric(reference, "concat", "lcc", "auc")
m0_hter = finite_metric(reference, "concat", "lcc", "hter")
m0_val_acer = finite_metric(reference, "concat", "val", "acer")
m0_apcer = finite_metric(reference, "concat", "lcc", "apcer")
m0_bpcer = finite_metric(reference, "concat", "lcc", "bpcer")
main_rows, ablation_rows, promising_ids = [], [], []
review = {
    "schema": "pad_micro_m0_m4_review_packet_v1",
    "lcc_dataset_role": "LCC-FASD external-dev / cross-domain stress set",
    "stage1_spatial_reference": {
        "summary_path": str(stage1_path),
        "checkpoint_sha256": stage1_sha,
        "val_acer": spatial_val_acer,
        "val_auc": spatial_val_auc,
        "lcc_auc": spatial_lcc_auc,
        "lcc_hter": spatial_lcc_hter,
        "lcc_candidate_n": int(stage1["lcc_candidate_n"]),
        "lcc_valid_face_n": int(stage1["lcc_valid_face_n"]),
    },
    "resource_hashes": reference_hashes,
    "source_summaries": summary_paths,
    "screening_heuristics": {
        "m0_role": "Frequency-representation control.",
        "stage1_role": "Spatial-only reference for whether frequency fusion helps spatial E1 at all.",
        "lcc_gain_vs_m0": "concat LCC AUC >= M0 + 0.020 OR (concat LCC HTER <= M0 - 0.020 AND AUC > M0)",
        "validation_guardrail": "concat micro-Val ACER regression <= 0.015",
        "branch_evidence": "FULL LCC AUC exceeds both FREQ_ZERO and FREQ_SHUFFLE by > 0.002",
        "spatial_match_tolerance": {"lcc_auc_abs_delta": 0.002, "lcc_hter_abs_delta": 0.005},
        "interpretation": "descriptive screening heuristics only; not statistical significance; no composite score",
        "borderline": "directional LCC AUC/HTER improvement >= 0.005 with micro-Val ACER regression <= 0.030, or larger gain with weak branch evidence",
    },
    "candidates": {},
}

for candidate_id in CANDIDATES:
    summary = summaries[candidate_id]
    freq_val_acer = finite_metric(summary, "frequency_only", "val", "acer")
    freq_val_auc = finite_metric(summary, "frequency_only", "val", "auc")
    freq_lcc_auc = finite_metric(summary, "frequency_only", "lcc", "auc")
    freq_lcc_hter = finite_metric(summary, "frequency_only", "lcc", "hter")
    concat_val_acer = finite_metric(summary, "concat", "val", "acer")
    concat_val_auc = finite_metric(summary, "concat", "val", "auc")
    concat_lcc_auc = finite_metric(summary, "concat", "lcc", "auc")
    concat_lcc_hter = finite_metric(summary, "concat", "lcc", "hter")
    concat_lcc_apcer = finite_metric(summary, "concat", "lcc", "apcer")
    concat_lcc_bpcer = finite_metric(summary, "concat", "lcc", "bpcer")
    full_auc = float(summary["branch_ablation"]["lcc"]["FULL"]["auc"])
    zero_auc = float(summary["branch_ablation"]["lcc"]["FREQ_ZERO"]["auc"])
    shuffle_auc = float(summary["branch_ablation"]["lcc"]["FREQ_SHUFFLE"]["auc"])
    delta_auc_m0 = concat_lcc_auc - m0_auc
    delta_hter_m0 = concat_lcc_hter - m0_hter
    val_regression_m0 = concat_val_acer - m0_val_acer
    delta_apcer_m0 = concat_lcc_apcer - m0_apcer
    delta_bpcer_m0 = concat_lcc_bpcer - m0_bpcer
    full_zero = full_auc - zero_auc
    full_shuffle = full_auc - shuffle_auc
    branch_supported = full_zero > 0.002 and full_shuffle > 0.002
    branch_weak = abs(full_zero) <= 0.002 or abs(full_shuffle) <= 0.002 or not branch_supported
    domain_gain = delta_auc_m0 >= 0.020 or (delta_hter_m0 <= -0.020 and delta_auc_m0 > 0.0)
    validation_ok = val_regression_m0 <= 0.015
    if candidate_id == "M0":
        status = "CONTROL"
        reason = "Frequency-representation control for all candidate deltas."
    elif domain_gain and validation_ok and branch_supported:
        status = "PROMISING"
        reason = "Meets descriptive LCC gain, micro-Val guardrail, and branch-evidence heuristics; this is not promotion."
        promising_ids.append(candidate_id)
    elif ((delta_auc_m0 >= 0.005 or delta_hter_m0 <= -0.005) and val_regression_m0 <= 0.030):
        status = "BORDERLINE"
        reason = "Some directional signal exists, but one or more screening criteria are not met."
    else:
        status = "KILL"
        reason = "No sufficient screening signal under the descriptive M0 heuristics."

    auc_delta_spatial = concat_lcc_auc - spatial_lcc_auc
    hter_delta_spatial = concat_lcc_hter - spatial_lcc_hter
    auc_relation = metric_relation(auc_delta_spatial, 0.002, higher_is_better=True)
    hter_relation = metric_relation(hter_delta_spatial, 0.005, higher_is_better=False)
    if auc_relation == "WITHIN_DESCRIPTIVE_TOLERANCE" and hter_relation == "WITHIN_DESCRIPTIVE_TOLERANCE":
        spatial_relation = "MATCHES_WITHIN_DESCRIPTIVE_TOLERANCE"
    elif auc_relation == "BETTER" and hter_relation == "BETTER":
        spatial_relation = "BEATS_ON_BOTH_AUC_AND_HTER"
    elif auc_relation == "WORSE" and hter_relation == "WORSE":
        spatial_relation = "BELOW_ON_BOTH_AUC_AND_HTER"
    else:
        spatial_relation = "MIXED_AUC_HTER_TRADE_OFF"

    if delta_hter_m0 < 0 and delta_auc_m0 <= 0:
        auc_hter_interpretation = "HTER_ONLY_OPERATING_POINT_TRADE_OFF; do not call a robust representation win"
    elif delta_auc_m0 > 0 and delta_hter_m0 < 0:
        auc_hter_interpretation = "AUC_AND_HTER_IMPROVE_VS_M0"
    elif delta_auc_m0 > 0:
        auc_hter_interpretation = "AUC_IMPROVES_BUT_THRESHOLD_SPECIFIC_HTER_DOES_NOT"
    elif delta_hter_m0 < 0:
        auc_hter_interpretation = "HTER_IMPROVES_WITHOUT_AUC_GAIN"
    else:
        auc_hter_interpretation = "NO_AUC_OR_HTER_GAIN_VS_M0"
    apcer_bpcer_tradeoff = (delta_apcer_m0 < 0 < delta_bpcer_m0) or (delta_bpcer_m0 < 0 < delta_apcer_m0)
    if apcer_bpcer_tradeoff:
        operating_point = "APCER_BPCER_TRADE_OFF_VS_M0"
    elif delta_apcer_m0 <= 0 and delta_bpcer_m0 <= 0:
        operating_point = "BOTH_ERRORS_NO_WORSE_VS_M0"
    else:
        operating_point = "NO_JOINT_APCER_BPCER_IMPROVEMENT_VS_M0"

    main_rows.append({
        "Candidate": candidate_id,
        "Freq Val ACER": freq_val_acer,
        "Freq Val AUC": freq_val_auc,
        "Freq LCC AUC": freq_lcc_auc,
        "Freq LCC HTER": freq_lcc_hter,
        "Concat Val ACER": concat_val_acer,
        "Concat Val AUC": concat_val_auc,
        "Concat LCC AUC": concat_lcc_auc,
        "Concat LCC HTER": concat_lcc_hter,
        "Concat LCC APCER": concat_lcc_apcer,
        "Concat LCC BPCER": concat_lcc_bpcer,
        "Delta LCC AUC vs M0": delta_auc_m0,
        "Delta LCC HTER vs M0": delta_hter_m0,
        "Delta LCC APCER vs M0": delta_apcer_m0,
        "Delta LCC BPCER vs M0": delta_bpcer_m0,
        "Spatial E1 Val ACER": spatial_val_acer,
        "Spatial E1 Val AUC": spatial_val_auc,
        "Spatial E1 LCC AUC": spatial_lcc_auc,
        "Spatial E1 LCC HTER": spatial_lcc_hter,
        "Concat Val ACER delta vs Spatial E1": concat_val_acer - spatial_val_acer,
        "Concat Val AUC delta vs Spatial E1": concat_val_auc - spatial_val_auc,
        "Concat LCC AUC delta vs Spatial E1": auc_delta_spatial,
        "Concat LCC HTER delta vs Spatial E1": hter_delta_spatial,
        "Spatial LCC AUC relation": auc_relation,
        "Spatial LCC HTER relation": hter_relation,
        "Spatial joint relation": spatial_relation,
        "AUC/HTER interpretation vs M0": auc_hter_interpretation,
        "APCER/BPCER interpretation vs M0": operating_point,
        "FULL - FREQ_ZERO Delta AUC": full_zero,
        "FULL - FREQ_SHUFFLE Delta AUC": full_shuffle,
        "Weak branch evidence": branch_weak,
        "Frequency params": int(summary["params"].get("frequency_branch", 0)),
        "Status": status,
    })
    review["candidates"][candidate_id] = {
        "status": status,
        "reason": reason,
        "beats_m0_auc": delta_auc_m0 > 0,
        "beats_m0_hter": delta_hter_m0 < 0,
        "beats_m0_on_both_auc_and_hter": delta_auc_m0 > 0 and delta_hter_m0 < 0,
        "delta_lcc_auc_vs_m0": delta_auc_m0,
        "delta_lcc_hter_vs_m0": delta_hter_m0,
        "spatial_reference": {
            "delta_lcc_auc_vs_spatial_e1": auc_delta_spatial,
            "lcc_auc_relation": auc_relation,
            "delta_lcc_hter_vs_spatial_e1": hter_delta_spatial,
            "lcc_hter_relation": hter_relation,
            "joint_relation": spatial_relation,
            "note": "FREQ_ZERO is a CONCAT-model ablation and is not the independent Stage 1 spatial baseline.",
        },
        "delta_lcc_apcer_vs_m0": delta_apcer_m0,
        "delta_lcc_bpcer_vs_m0": delta_bpcer_m0,
        "apcer_bpcer_tradeoff_vs_m0": apcer_bpcer_tradeoff,
        "auc_hter_interpretation_vs_m0": auc_hter_interpretation,
        "concat_val_acer_regression_vs_m0": val_regression_m0,
        "full_minus_freq_zero_lcc_auc": full_zero,
        "full_minus_freq_shuffle_lcc_auc": full_shuffle,
        "weak_branch_evidence": branch_weak,
        "branch_evidence_supported": branch_supported,
    }
    for domain in ["val", "lcc"]:
        for mode, values in summary["branch_ablation"][domain].items():
            ablation_rows.append({
                "Candidate": candidate_id, "Domain": domain, "Mode": mode,
                "AUC": values.get("auc"), "HTER": values.get("hter"),
                "ACER": values.get("acer"), "APCER": values.get("apcer"),
                "BPCER": values.get("bpcer"), "Accuracy": values.get("accuracy"),
            })

review["overall"] = {
    "screening_status": "SCREENING_ONLY",
    "winner_status": "NO WINNER",
    "promising_candidates_for_later_review": promising_ids,
    "composite_score": None,
    "statistical_significance_claimed": False,
    "automatic_promotion": False,
}
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
main_df = pd.DataFrame(main_rows)
main_df.to_csv(out / "micro_m0_m4_comparison.csv", index=False)
pd.DataFrame(ablation_rows).to_csv(out / "micro_m0_m4_ablation_comparison.csv", index=False)
(out / "micro_m0_m4_review_packet.json").write_text(json.dumps(review, indent=2, allow_nan=False) + "\n")
print(main_df.to_string(index=False))
print("\nSpatial E1 baseline loaded from:", stage1_path)
print("Review status:", review["overall"]["winner_status"])
print("Comparison artifacts:", out)
print("Raw CelebA/LCC image datasets are not required for this review notebook.")
print("These are screening heuristics, not significance tests or final claims.")


Candidate  Freq Val ACER  Freq Val AUC  Freq LCC AUC  Freq LCC HTER  Concat Val ACER  Concat Val AUC  Concat LCC AUC  Concat LCC HTER  Concat LCC APCER  Concat LCC BPCER  Delta LCC AUC vs M0  Delta LCC HTER vs M0  Delta LCC APCER vs M0  Delta LCC BPCER vs M0  Spatial E1 Val ACER  Spatial E1 Val AUC  Spatial E1 LCC AUC  Spatial E1 LCC HTER  Concat Val ACER delta vs Spatial E1  Concat Val AUC delta vs Spatial E1  Concat LCC AUC delta vs Spatial E1  Concat LCC HTER delta vs Spatial E1 Spatial LCC AUC relation    Spatial LCC HTER relation     Spatial joint relation AUC/HTER interpretation vs M0       APCER/BPCER interpretation vs M0  FULL - FREQ_ZERO Delta AUC  FULL - FREQ_SHUFFLE Delta AUC  Weak branch evidence  Frequency params  Status
       M0       0.228652      0.829130      0.444751       0.522232         0.020277        0.997434        0.685769         0.352281          0.411503          0.293059             0.000000              0.000000               0.000000               0.0000